# Moderación semiautomática de videos peruanos de YouTube mediante modelos clásicos y neuronales de procesamiento del lenguaje natural

**Trabajo final del curso de Procesamiento de Lenguaje Natural (PLN) de la Maestría en Inteligencia Artificial de la Universidad Nacional de Ingeniería (UNI) — Semestre 2026-1**

**Grupo 4:** Luis Enrique Koc Góngora, Alex Felipe Mancilla Antay, Herbert Antonio Meléndez García y Dennis Jack Paitán Cano

---

## 02.01 · Cascada de etiquetado calibrada

Reproduce el patrón histórico Flash→Pro: calibra una primera pasada económica, etiqueta el corpus por lotes y dirige los casos riesgosos a un revisor más capaz.

La procedencia de `deepseek-v4-flash` y `deepseek-v4-pro` está documentada por el proveedor [1], al igual que sus precios por tokens y caché [2]. La selección dirigida pertenece a la familia de aprendizaje activo [3]. El acuerdo Flash–Pro calibra una regla operativa, pero no constituye *ground truth*; las tareas subjetivas conservan una instancia humana final independiente [4]. Los umbrales, el presupuesto, el control seguro y la precedencia son decisiones locales auditables.

**Contrato de etiquetas v2.1:** cinco salidas entrenadas: `SEGURO`, `RACISMO_DISCRIMINACION`, `ATAQUE_POR_GENERO_IDENTIDAD`, `ACOSO_AMENAZA` y `CONTENIDO_SEXUAL`. `SEGURO` es excluyente; las cuatro categorías de daño son multietiqueta y pueden coexistir. Los casos indeterminados se difieren y no entran al entrenamiento. Esta combinación, sus umbrales y sus reglas de exclusividad son decisiones operativas locales.

## Reproducibilidad

El cuaderno solo orquesta funciones versionadas de `src/moderacion_peru`. En local no instala paquetes. En Colab, únicamente la celda de bootstrap instala versiones fijadas desde el bundle SHA-256 de Drive. No usa rutas personales. Revise el README de esta etapa.

## Backend opcional Google Colab desde VS Code

Instale la extensión oficial **Google Colab** (`google.colab`), seleccione `Select Kernel > Colab`; esta campaña API funciona con runtime CPU. El notebook permanece local; Drive transporta solo versiones inmutables del bundle. La celda detecta si falta el release exacto: en ese único caso lo obtiene desde GitHub —o mediante `local_upload`—, verifica todos sus SHA-256 y lo publica de forma atómica. Después promueve la copia activa cuando sea necesario; ya no requiere ejecutar `02_00` previamente. Edite `COLAB_RUN_ID` para separar experimentos. La compatibilidad de `drive.mount()` desde VS Code requiere la extensión v0.2.1 o posterior [5]. La integridad del bundle se comprueba con SHA-256 [6]. No sincronice cachés de modelos ni entrene directamente sobre Drive; los cuadernos de entrenamiento Qwen copian cada checkpoint terminado mediante un TAR atómico y reanudable.

In [ ]:
# Backend reproducible: local o Google Colab desde VS Code
from datetime import datetime, timezone
from pathlib import Path
import hashlib
import importlib.util
import json
import os
import shutil
import subprocess
import sys
import urllib.parse
import urllib.request
import uuid
import zipfile

COLAB_NOTEBOOK_ID = "02_01"
COLAB_DRIVE_FOLDER = "ModeracionPeru_Colab"  # Debe coincidir con config/colab_l4.json
COLAB_RUN_ID = ""  # Vacío reanuda <notebook>_working_v2_1; use otro ID para otro experimento
COLAB_REQUIRE_L4 = True
COLAB_AUTO_UPDATE_BUNDLE = True
COLAB_AUTO_PUBLISH_MISSING_BUNDLE = True
COLAB_BUNDLE_SOURCE = "github"  # "github" o "local_upload"
COLAB_GITHUB_REPOSITORY = "lkoc/Trabajo_PLN-MIA-Grupo4"
COLAB_GITHUB_REF = "main"
COLAB_GITHUB_BUNDLE_PATH = "resultados/colab_bundle"
COLAB_NOTEBOOK_BUILD_BUNDLE_ID = "57820ed6c4b2453e53cefb1fde9b8c4675b22bdf21db0159eaec08c315506691"  # Trazabilidad al generar el notebook
COLAB_EXPECTED_CORE_SHA256 = "cded349ce51da1421aab4a13f40de61bc471674151bf6a0b6178198b397ad1f2"
IN_COLAB = importlib.util.find_spec("google.colab") is not None
COLAB_CONTEXT = None

# Los modelos configurados son públicos. Evita que huggingface_hub intente
# consultar el vault de secretos, que solo funciona desde la interfaz web de Colab.
if IN_COLAB:
    os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
    os.environ["HF_HOME"] = "/content/huggingface"

def _sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while block := handle.read(1024 * 1024):
            digest.update(block)
    return digest.hexdigest()

def _find_local_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError("No se encontró pyproject.toml")

def _read_manifest(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def _bundle_id_for_manifest(manifest):
    core = manifest["core"]
    inputs = manifest["inputs"]
    identity = {
        "schema_version": manifest["schema_version"],
        "taxonomy_contract": manifest["taxonomy_contract"],
        "taxonomy_version": manifest["taxonomy_version"],
        "core": {"name": core["name"], "sha256": core["sha256"]},
        "inputs": {
            key: {
                "archive": value["archive"],
                "archive_sha256": value["archive_sha256"],
                "source_sha256": value["source_sha256"],
            }
            for key, value in sorted(inputs.items())
        },
    }
    payload = json.dumps(identity, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

def _bundle_specs(manifest):
    specs = [(manifest["core"]["name"], manifest["core"]["sha256"])]
    specs.extend(
        (entry["archive"], entry["archive_sha256"])
        for entry in manifest.get("inputs", {}).values()
    )
    for name, expected_sha256 in specs:
        if Path(name).name != name or not expected_sha256:
            raise ValueError(f"Entrada insegura o incompleta en bundle_manifest.json: {name!r}")
    return specs

def _verify_expected_bundle(bundle_dir, expected_bundle_id=COLAB_NOTEBOOK_BUILD_BUNDLE_ID):
    manifest_path = Path(bundle_dir) / "bundle_manifest.json"
    if not manifest_path.is_file():
        raise FileNotFoundError(f"Falta {manifest_path}")
    manifest = _read_manifest(manifest_path)
    computed_bundle_id = _bundle_id_for_manifest(manifest)
    if manifest.get("bundle_id") != computed_bundle_id:
        raise ValueError("bundle_manifest.json no contiene una identidad válida")
    if computed_bundle_id != expected_bundle_id:
        raise ValueError(
            f"Bundle inesperado: esperado={expected_bundle_id}, obtenido={computed_bundle_id}"
        )
    if manifest["core"]["sha256"] != COLAB_EXPECTED_CORE_SHA256:
        raise ValueError("El core del bundle no coincide con el fijado por este cuaderno")
    for name, expected_sha256 in _bundle_specs(manifest):
        artifact = Path(bundle_dir) / name
        if not artifact.is_file() or _sha256(artifact) != expected_sha256:
            raise ValueError(f"Artefacto ausente o inválido: {artifact}")
    return manifest

def _bundle_is_current(bundle_dir, manifest_path, expected_bundle_id):
    if Path(manifest_path) != Path(bundle_dir) / "bundle_manifest.json":
        return False
    try:
        _verify_expected_bundle(bundle_dir, expected_bundle_id)
        return True
    except (OSError, KeyError, TypeError, ValueError, json.JSONDecodeError):
        return False

def _download_bundle_file(url, destination):
    destination = Path(destination)
    partial = destination.with_name(f".{destination.name}.partial")
    request = urllib.request.Request(
        url,
        headers={"User-Agent": "ModeracionPeru-Colab-Bundle/2.0"},
    )
    try:
        with urllib.request.urlopen(request, timeout=180) as response, partial.open("wb") as target:
            while block := response.read(1024 * 1024):
                target.write(block)
        os.replace(partial, destination)
    finally:
        if partial.exists():
            partial.unlink()

def _prepare_bundle_staging():
    staging = Path("/content/moderacion_peru_bundle_source")
    if staging.exists():
        shutil.rmtree(staging)
    staging.mkdir(parents=True)
    return staging

def _acquire_expected_bundle():
    staging = _prepare_bundle_staging()
    if COLAB_BUNDLE_SOURCE == "github":
        encoded_ref = urllib.parse.quote(COLAB_GITHUB_REF, safe="")
        base = (
            f"https://raw.githubusercontent.com/{COLAB_GITHUB_REPOSITORY}/"
            f"{encoded_ref}/{COLAB_GITHUB_BUNDLE_PATH}"
        )
        manifest_path = staging / "bundle_manifest.json"
        _download_bundle_file(f"{base}/bundle_manifest.json", manifest_path)
        manifest = _read_manifest(manifest_path)
        if manifest.get("bundle_id") != _bundle_id_for_manifest(manifest):
            raise ValueError("El manifiesto descargado desde GitHub no es válido")
        if manifest["bundle_id"] != COLAB_NOTEBOOK_BUILD_BUNDLE_ID:
            raise RuntimeError(
                "GitHub todavía no contiene el bundle fijado por este cuaderno. "
                "Sincronice resultados/colab_bundle o use COLAB_BUNDLE_SOURCE='local_upload'."
            )
        if manifest["core"]["sha256"] != COLAB_EXPECTED_CORE_SHA256:
            raise RuntimeError("GitHub contiene un project_core.zip distinto al esperado")
        for name, _ in _bundle_specs(manifest):
            encoded_name = urllib.parse.quote(name, safe="")
            _download_bundle_file(f"{base}/{encoded_name}", staging / name)
    elif COLAB_BUNDLE_SOURCE == "local_upload":
        from google.colab import files

        uploaded = files.upload()
        if "bundle_manifest.json" not in uploaded:
            raise FileNotFoundError("La selección no incluyó bundle_manifest.json")
        (staging / "bundle_manifest.json").write_bytes(uploaded["bundle_manifest.json"])
        manifest = _read_manifest(staging / "bundle_manifest.json")
        if manifest.get("bundle_id") != COLAB_NOTEBOOK_BUILD_BUNDLE_ID:
            raise RuntimeError("Los archivos seleccionados no pertenecen al bundle esperado")
        required = {"bundle_manifest.json", *(name for name, _ in _bundle_specs(manifest))}
        missing = sorted(required - set(uploaded))
        if missing:
            raise FileNotFoundError(f"Faltaron archivos del bundle: {missing}")
        for name in required - {"bundle_manifest.json"}:
            (staging / name).write_bytes(uploaded[name])
    else:
        raise ValueError("COLAB_BUNDLE_SOURCE debe ser 'github' o 'local_upload'")
    return staging, _verify_expected_bundle(staging)

def _write_latest_pointer(releases_dir, release_dir, manifest):
    pointer = {
        "schema_version": "1.0.0",
        "bundle_id": manifest["bundle_id"],
        "core_sha256": manifest["core"]["sha256"],
        "manifest_sha256": _sha256(Path(release_dir) / "bundle_manifest.json"),
        "published_at": datetime.now(timezone.utc).isoformat(),
    }
    latest_path = Path(releases_dir) / "latest.json"
    partial = Path(releases_dir) / f".latest-{uuid.uuid4().hex}.json"
    partial.write_text(json.dumps(pointer, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    os.replace(partial, latest_path)
    return pointer

def _publish_expected_bundle(staging, releases_dir):
    manifest = _verify_expected_bundle(staging)
    releases_dir = Path(releases_dir)
    releases_dir.mkdir(parents=True, exist_ok=True)
    release_dir = releases_dir / COLAB_NOTEBOOK_BUILD_BUNDLE_ID
    if release_dir.exists():
        _verify_expected_bundle(release_dir)
        release_status = "already_present_and_verified"
    else:
        partial = releases_dir / f".{COLAB_NOTEBOOK_BUILD_BUNDLE_ID}.partial-{uuid.uuid4().hex}"
        partial.mkdir()
        try:
            for name, _ in _bundle_specs(manifest):
                shutil.copyfile(Path(staging) / name, partial / name)
            shutil.copyfile(
                Path(staging) / "bundle_manifest.json",
                partial / "bundle_manifest.json",
            )
            _verify_expected_bundle(partial)
            os.replace(partial, release_dir)
        finally:
            if partial.exists():
                shutil.rmtree(partial)
        release_status = "auto_published_and_verified"
    pointer = _write_latest_pointer(releases_dir, release_dir, manifest)
    return {
        "status": release_status,
        "release_dir": release_dir,
        "latest_pointer": pointer,
    }

def _ensure_expected_drive_release(releases_dir):
    release_dir = Path(releases_dir) / COLAB_NOTEBOOK_BUILD_BUNDLE_ID
    if _bundle_is_current(
        release_dir,
        release_dir / "bundle_manifest.json",
        COLAB_NOTEBOOK_BUILD_BUNDLE_ID,
    ):
        return {"status": "already_present_and_verified", "release_dir": release_dir}
    if not COLAB_AUTO_PUBLISH_MISSING_BUNDLE:
        raise RuntimeError(
            "Drive no contiene el release esperado y COLAB_AUTO_PUBLISH_MISSING_BUNDLE=False"
        )
    staging, _ = _acquire_expected_bundle()
    return _publish_expected_bundle(staging, releases_dir)

def _activate_verified_drive_release(release_dir, bundle_dir, expected_bundle_id):
    release_manifest_path = release_dir / "bundle_manifest.json"
    if not _bundle_is_current(release_dir, release_manifest_path, expected_bundle_id):
        raise RuntimeError(
            "La versión esperada no está completa o no coincide con sus SHA-256: " + str(release_dir)
        )
    manifest = _read_manifest(release_manifest_path)
    bundle_dir.mkdir(parents=True, exist_ok=True)
    # Todos los artefactos se validaron antes; el manifiesto activo se reemplaza al final.
    for name, _ in _bundle_specs(manifest):
        partial = bundle_dir / f".{name}.partial"
        shutil.copyfile(release_dir / name, partial)
        os.replace(partial, bundle_dir / name)
    partial_manifest = bundle_dir / ".bundle_manifest.json.partial"
    shutil.copyfile(release_manifest_path, partial_manifest)
    os.replace(partial_manifest, bundle_dir / "bundle_manifest.json")
    if not _bundle_is_current(bundle_dir, bundle_dir / "bundle_manifest.json", expected_bundle_id):
        raise RuntimeError("La activación desde bundle_releases no superó la verificación final")
    return manifest

if IN_COLAB:
    from google.colab import drive

    # La extensión oficial de Colab para VS Code admite drive.mount desde v0.2.1.
    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = Path("/content/drive/MyDrive") / COLAB_DRIVE_FOLDER
    BUNDLE_DIR = DRIVE_ROOT / "bundle"
    RELEASES_DIR = DRIVE_ROOT / "bundle_releases"
    RELEASES_DIR.mkdir(parents=True, exist_ok=True)
    release_check = _ensure_expected_drive_release(RELEASES_DIR)
    latest_bundle_id = COLAB_NOTEBOOK_BUILD_BUNDLE_ID
    RELEASE_DIR = RELEASES_DIR / latest_bundle_id
    manifest = _verify_expected_bundle(RELEASE_DIR)
    release_manifest_path = RELEASE_DIR / "bundle_manifest.json"
    latest_pointer_path = RELEASES_DIR / "latest.json"
    latest_pointer = _read_manifest(latest_pointer_path) if latest_pointer_path.is_file() else {}
    latest_matches_notebook = (
        latest_pointer.get("bundle_id") == COLAB_NOTEBOOK_BUILD_BUNDLE_ID
        and latest_pointer.get("core_sha256") == COLAB_EXPECTED_CORE_SHA256
        and latest_pointer.get("manifest_sha256") == _sha256(release_manifest_path)
    )
    if latest_matches_notebook:
        release_source = (
            "auto_published_from_" + COLAB_BUNDLE_SOURCE
            if release_check["status"] == "auto_published_and_verified"
            else "latest_pointer"
        )
    else:
        # Un cuaderno reproducible puede activar su release inmutable exacto aunque
        # latest todavía apunte a otra versión; jamás mezcla código e inputs.
        release_source = "notebook_pinned_release"
    manifest_path = BUNDLE_DIR / "bundle_manifest.json"
    bundle_activated = False
    modules_loaded_before_update = any(
        name == "moderacion_peru" or name.startswith("moderacion_peru.") for name in sys.modules
    )
    if not _bundle_is_current(BUNDLE_DIR, manifest_path, latest_bundle_id):
        if not COLAB_AUTO_UPDATE_BUNDLE:
            raise RuntimeError("El bundle de Drive está desactualizado y COLAB_AUTO_UPDATE_BUNDLE=False")
        try:
            manifest = _activate_verified_drive_release(RELEASE_DIR, BUNDLE_DIR, latest_bundle_id)
            bundle_activated = True
        except Exception as exc:
            raise RuntimeError(
                "No fue posible activar la versión esperada desde Google Drive después de "
                f"verificar o autopublicar {RELEASE_DIR}."
            ) from exc
    else:
        manifest = _read_manifest(manifest_path)

    core = BUNDLE_DIR / manifest["core"]["name"]
    if _sha256(core) != manifest["core"]["sha256"]:
        raise ValueError("project_core.zip no coincide con el manifiesto SHA-256")

    RUNTIME_ROOT = Path("/content/moderacion_peru")
    ROOT = RUNTIME_ROOT / "project"
    marker = RUNTIME_ROOT / ".core_sha256"
    expected_core = manifest["core"]["sha256"]
    if not ROOT.is_dir() or not marker.is_file() or marker.read_text().strip() != expected_core:
        if ROOT.exists():
            shutil.rmtree(ROOT)
        ROOT.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(core) as archive:
            archive.extractall(ROOT)
        os.environ["PIP_DISABLE_PIP_VERSION_CHECK"] = "1"
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements/colab-l4.txt")]
        )
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(ROOT)])
        marker.parent.mkdir(parents=True, exist_ok=True)
        marker.write_text(expected_core + "\n", encoding="utf-8")

    if bundle_activated and modules_loaded_before_update:
        raise RuntimeError(
            "El bundle se actualizó y verificó en Drive, pero este kernel ya había importado una "
            "versión anterior de moderacion_peru. Reinicie completamente el kernel de Colab y vuelva "
            "a ejecutar el cuaderno desde la primera celda."
        )

    os.environ["MODPERU_ROOT"] = str(ROOT)
    importlib.invalidate_caches()
    if str(ROOT / "src") not in sys.path:
        sys.path.insert(0, str(ROOT / "src"))
    from moderacion_peru.colab import colab_runtime_diagnostics, prepare_colab_context

    COLAB_CONTEXT = prepare_colab_context(
        COLAB_NOTEBOOK_ID,
        project_root=ROOT,
        drive_root=DRIVE_ROOT,
        runtime_root=RUNTIME_ROOT,
        run_id=COLAB_RUN_ID or None,
        require_l4=COLAB_REQUIRE_L4,
        resume=True,
    )
    from moderacion_peru.notebook_ui import notebook_progress, run_with_progress, show_callout, show_command, show_result, show_summary, show_table
    show_result('Bundle de Colab verificado', {
        'estado': 'activado_desde_drive' if bundle_activated else 'ya_estaba_actualizado',
        'bundle_id': manifest['bundle_id'],
        'bundle_del_notebook_al_generarse': COLAB_NOTEBOOK_BUILD_BUNDLE_ID,
        'origen_del_release': release_source,
        'estado_del_release': release_check['status'],
        'core_sha256': expected_core,
        'generado': manifest.get('generated_at'),
        'versión_inmutable_drive': RELEASE_DIR,
    }, tone='success')
    show_result('Diagnóstico de Colab', colab_runtime_diagnostics(), tone='success')
    show_result('Contexto reproducible', COLAB_CONTEXT.as_dict(), tone='success')
else:
    ROOT = _find_local_root()
    if str(ROOT / "src") not in sys.path:
        sys.path.insert(0, str(ROOT / "src"))
    from moderacion_peru.notebook_ui import notebook_progress, run_with_progress, show_callout, show_command, show_result, show_summary, show_table
    show_summary('Entorno del proyecto', {'raíz': ROOT, 'backend': 'local'}, tone='success')
OPERATIONAL_PROMPT=ROOT/'config/prompt_operacional_ollama_v3_2.md'
if not OPERATIONAL_PROMPT.is_file():
    raise FileNotFoundError(f'Falta el prompt operacional vigente: {OPERATIONAL_PROMPT}')
show_summary('Prompt operacional vigente', {'ruta': OPERATIONAL_PROMPT, 'versión': '3.2.0'}, tone='success')


## Configuración explícita y credencial

In [ ]:
import os
from pathlib import Path
from moderacion_peru.providers import DeepSeekProvider

if globals().get('IN_COLAB') and not os.getenv('DEEPSEEK_API_KEY'):
    from google.colab import userdata

    try:
        os.environ['DEEPSEEK_API_KEY']=userdata.get('DEEPSEEK_API_KEY') or ''
    except Exception:
        pass

SOURCE=COLAB_CONTEXT.input('chunks_v2') if COLAB_CONTEXT else ROOT/'datos/processed/chunks_v2.jsonl'
CAMPAIGN_ROOT=COLAB_CONTEXT.scratch_output_dir if COLAB_CONTEXT else ROOT/'datos/etiquetado/cascada_deepseek_v4'
CAMPAIGN_ROOT.mkdir(parents=True,exist_ok=True)
HISTORICAL_CHUNKS=COLAB_CONTEXT.input('chunks_deepseek_historicos') if COLAB_CONTEXT else ROOT/'datos/processed/chunks_para_etiquetar.jsonl'
HISTORICAL_FLASH_SOURCES=(COLAB_CONTEXT.input('deepseek_flash_historico'),) if COLAB_CONTEXT else (ROOT/'datos/etiquetado/llm_api/deepseek-v4-flash_labeled_chunks_seed42.jsonl',)
HISTORICAL_PRO_SOURCES=(COLAB_CONTEXT.input('deepseek_pro_historico_principal'),COLAB_CONTEXT.input('deepseek_pro_historico_umbral'),COLAB_CONTEXT.input('deepseek_pro_historico_sospechosos')) if COLAB_CONTEXT else (ROOT/'datos/etiquetado/llm_api/deepseek-v4-pro_revision_de_deepseek-v4-flash_seed42.jsonl',ROOT/'datos/etiquetado/llm_api/deepseek-v4-pro_revision_umbral_recalibrado_t090_seed42.jsonl',ROOT/'datos/etiquetado/llm_api/deepseek-v4-pro_revision_sospechosos_gruesos_seed42.jsonl')
HISTORICAL_PROMPT_SHA256='52d4fec14ad433d35ec20de5f51a6954aad69dcedd1422059419dcecc2f9e778'
PRESERVE_PROMPT_POLICY_SHA256='433321cf7b41f997bb277ae87bc9fee01767d225a0fe49bea2cb918239dc1f06'
PREVIOUS_PRIMARY_PATH=CAMPAIGN_ROOT/'primary_flash.jsonl'
PREVIOUS_REVIEW_PATH=CAMPAIGN_ROOT/'review_pro.jsonl'
PRIMARY_PATH=CAMPAIGN_ROOT/'primary_flash_v3_2.jsonl'
REVIEW_PATH=CAMPAIGN_ROOT/'review_pro_v3_2.jsonl'
RECOVER_HISTORICAL=True  # Recupera solo coincidencias exactas 1:1; nunca transfiere segmentos distintos.
AUTO_PUBLISH_CHECKPOINTS=True  # En Colab publica TAR.GZ atómico al recuperar, periódicamente y al interrumpir.
DRIVE_CHECKPOINT_EVERY_BATCHES=3  # 3 ventanas × 640 chunks; cada grupo de 5 ya queda fsync local.
RUN_API_PREFLIGHT=True  # Consulta /models y /user/balance sin enviar textos ni consumir tokens de etiquetado.
RUN_CALIBRATION=False  # Primero: panel pareado Flash–Pro. Costo esperado muy bajo.
RUN_PRIMARY=True  # Procesa únicamente los chunk_id nuevos o todavía pendientes en Flash.
RUN_DIRECTED_REVIEW=True  # Reanuda Pro únicamente sobre la cola presupuestada pendiente.
CALIBRATION_PANEL_SIZE=1000  # Aún breve; permite evaluar LI95%≈0.95 con potencia útil.
PRIMARY_LIMIT=None  # None para TODOS y solo los pendientes; use 20 únicamente para un smoke test. Nunca lo deje en blanco.
REVIEW_LIMIT=None   # Campaña: TODA y solo la cola dirigida pendiente.
PROCESSING_BATCH_SIZE=640  # Ventana persistible: 128 solicitudes de 5; Pro limita el fan-out a 64.
MAX_PRIMARY_COST_USD=60.0
MAX_REVIEW_COST_USD=None  # Sin bloqueo artificial: el saldo real y los checkpoints gobiernan la reanudación.
BALANCE_REFRESH_SECONDS=60.0  # Consulta de saldo sin corpus durante la ejecución.
LOW_BALANCE_WARNING_USD=2.0
CACHE_ALERT_AFTER_REQUESTS=50
MIN_CACHE_HIT_RATE=0.50
SAFE_CONTROL_RATE=0.01  # Control seguro aleatorio reproducible del 1%.
REVIEW_CONFIDENCE_THRESHOLD=0.85  # Pro revisa seguros Flash < 0.85; el 0.95 sigue siendo solo diagnóstico.
MAX_NEEDS_REVIEW_FOR_PRO=36_000  # Abstenciones Flash de menor confianza; desempate SHA-256 reproducible.

OBSERVED_REVIEW_COST_PER_1000_USD=0.335785  # Tramo Pro medido: US$4.727523 / 14 079 respuestas.
RECOMMENDED_REVIEW_START_BALANCE_USD=15.00  # Solo advertencia; no bloquea una corrida con saldo menor.

primary_provider=DeepSeekProvider(model='deepseek-v4-flash',max_workers=128,records_per_request=5,cache_warmup_requests=1,max_cost_usd=MAX_PRIMARY_COST_USD,label_source='deepseek_remote',operational_prompt_path=OPERATIONAL_PROMPT)
reviewer_provider=DeepSeekProvider(model='deepseek-v4-pro',max_workers=64,records_per_request=5,cache_warmup_requests=1,max_cost_usd=MAX_REVIEW_COST_USD,label_source='llm_remote_review',operational_prompt_path=OPERATIONAL_PROMPT)
primary_probe=primary_provider.probe(); reviewer_probe=reviewer_provider.probe()
expected_thinking={'type':'disabled'}
expected_response_format={'type':'json_object'}
expected_cache_usage_fields=['prompt_cache_hit_tokens','prompt_cache_miss_tokens']
if primary_probe['thinking'] != expected_thinking or reviewer_probe['thinking'] != expected_thinking:
    raise RuntimeError('02_01 exige DeepSeek V4 en modo non-thinking para Flash y Pro')
if primary_probe['response_format'] != expected_response_format or reviewer_probe['response_format'] != expected_response_format or primary_probe['output_contract']['root_key'] != 'annotations' or reviewer_probe['output_contract']['root_key'] != 'annotations':
    raise RuntimeError('02_01 exige JSON object con raíz annotations para Flash y Pro')
if primary_probe['context_cache']['mode'] != 'automatic_prefix' or reviewer_probe['context_cache']['mode'] != 'automatic_prefix':
    raise RuntimeError('02_01 exige caché de contexto automática con prefijo estable')
if primary_probe['context_cache']['verified_from_usage_fields'] != expected_cache_usage_fields or reviewer_probe['context_cache']['verified_from_usage_fields'] != expected_cache_usage_fields:
    raise RuntimeError('02_01 debe medir aciertos y fallos reales de caché en la respuesta de DeepSeek')
if not primary_probe['credential_configured']:
    show_callout('Falta credencial','Defina DEEPSEEK_API_KEY en el entorno o como secreto de Colab. El preflight no consume crédito.',tone='warning')
show_result('Primera pasada',primary_probe,tone='success')
show_result('Revisor dirigido',reviewer_probe,tone='success')
show_summary('Modo DeepSeek verificado',{'Flash':primary_probe['thinking'],'Pro':reviewer_probe['thinking'],'JSON_Flash':primary_probe['response_format'],'JSON_Pro':reviewer_probe['response_format'],'contrato_salida':primary_probe['output_contract'],'caché_Flash':primary_probe['context_cache'],'caché_Pro':reviewer_probe['context_cache'],'lote':5,'concurrencia_Flash':primary_probe['max_workers'],'concurrencia_Pro':reviewer_probe['max_workers']},tone='success')
if RUN_API_PREFLIGHT and primary_probe['credential_configured']:
    flash_connection=primary_provider.validate_connection(); pro_connection=reviewer_provider.validate_connection()
    if not flash_connection['model_available'] or not pro_connection['model_available']:
        raise RuntimeError('Flash o Pro no aparece disponible en el catálogo de DeepSeek')
    initial_balance=primary_provider.balance_summary()
    show_result('Credencial, modelos y saldo verificados; no se enviaron textos',{'Flash':flash_connection,'Pro':pro_connection,'saldo':initial_balance},tone='success' if initial_balance['is_available'] else 'warning')
show_summary('Rutas y activación',{'entrada':SOURCE,'campaña':CAMPAIGN_ROOT,'recuperación_histórica':RECOVER_HISTORICAL,'checkpoint_drive_automático':bool(COLAB_CONTEXT and AUTO_PUBLISH_CHECKPOINTS),'calibración':RUN_CALIBRATION,'primaria':RUN_PRIMARY,'revisión':RUN_DIRECTED_REVIEW,'umbral_seguro_Flash_a_Pro':REVIEW_CONFIDENCE_THRESHOLD,'máximo_abstenciones_Pro':MAX_NEEDS_REVIEW_FOR_PRO,'control_seguro':SAFE_CONTROL_RATE,'tope_reanudación_USD':MAX_REVIEW_COST_USD},tone='neutral')

## Carga visible del corpus

In [3]:
if globals().get('IN_COLAB'):
    from tqdm.std import tqdm  # Salida textual visible también desde VS Code.
else:
    from tqdm.auto import tqdm
from moderacion_peru.io import read_jsonl
CHUNKS=list(tqdm(read_jsonl(SOURCE),desc='Cargando chunks',unit='chunk'))
show_summary('Corpus disponible',{'chunks_totales':len(CHUNKS),'nota':'La recuperación histórica se ejecuta antes de calcular lo pendiente y su costo.'},tone='neutral')

Cargando chunks: 0chunk [00:00, ?chunk/s]

chunks_totales,166940
nota,La recuperación histórica se ejecuta antes de calcular lo pendiente y su costo.


## Funciones de ejecución, avance y checkpoint

In [ ]:
from moderacion_peru.io import read_jsonl,write_json_atomic,write_jsonl_atomic
from moderacion_peru.labeling import annotate_batched_incremental,historical_recovery_signature,recover_historical_annotations
import time
if COLAB_CONTEXT is not None:
    from moderacion_peru.colab import publish_colab_outputs

def labeling_progress(description,provider):
    state={'bar':None,'last_balance_check':0.0,'balance':None,'balance_error':None,'cache_alerted':False,'low_balance_alerted':False}
    def refresh_balance(*,force=False):
        now=time.monotonic()
        if not force and now-state['last_balance_check'] < BALANCE_REFRESH_SECONDS: return
        state['last_balance_check']=now
        try:
            state['balance']=provider.balance_summary(); state['balance_error']=None
            total=state['balance']['total_balance_usd']
            if total <= LOW_BALANCE_WARNING_USD and not state['low_balance_alerted']:
                tqdm.write(f'⚠ Saldo DeepSeek bajo: US${total:.2f}. Considere recargar antes de continuar.')
                state['low_balance_alerted']=True
            elif total > LOW_BALANCE_WARNING_USD:
                state['low_balance_alerted']=False
        except Exception as exc:
            message=f'{type(exc).__name__}: {exc}'
            if message != state['balance_error']: tqdm.write(f'⚠ No se pudo actualizar el saldo DeepSeek: {message}')
            state['balance_error']=message
    def update_postfix(event):
        bar=state.get('bar'); usage=event.get('provider_usage') or {}; cache=usage.get('cache_hit_rate')
        balance=state.get('balance') or {}; total=balance.get('total_balance_usd')
        if bar is not None:
            bar.set_postfix(ok=event.get('labeled',0),errores=event.get('errors',0),gastado_USD=f"{usage.get('estimated_cost_usd',0):.4f}",saldo_USD='—' if total is None else f'{total:.2f}',caché='—' if cache is None else f'{100*cache:.1f}%')
        if usage.get('requests',0) >= CACHE_ALERT_AFTER_REQUESTS and cache is not None and cache < MIN_CACHE_HIT_RATE and not state['cache_alerted']:
            tqdm.write(f'⚠ Caché DeepSeek baja ({100*cache:.1f}%). Revise antes de ampliar la campaña; el progreso ya guardado no se pierde.')
            state['cache_alerted']=True
    def callback(event):
        if event['status']=='phase_started':
            if state.get('bar') is not None: state['bar'].close()
            label='Verificando progreso guardado' if event['phase']=='existing_progress' else 'Buscando chunks pendientes'
            state['bar']=tqdm(total=event.get('total'),desc=label,unit='chunk'); return
        if event['status']=='phase_progress':
            if state.get('bar') is not None: state['bar'].update(event.get('phase_advance',0))
            return
        if event['status']=='phase_finished':
            if state.get('bar') is not None: state['bar'].close(); state['bar']=None
            return
        if event['status']=='started':
            state['bar']=tqdm(total=event['selected'],desc=description,unit='chunk')
            refresh_balance(force=True); update_postfix(event)
            if state.get('balance') is not None and not state['balance']['is_available']:
                state['bar'].close(); state['bar']=None
                raise RuntimeError(f"DeepSeek no tiene saldo disponible (US${state['balance']['total_balance_usd']:.2f}); recargue y vuelva a ejecutar. No se envió ningún chunk pendiente.")
            return
        bar=state.get('bar')
        if bar is not None and event.get('advance'):
            bar.update(event['advance'])
            refresh_balance(); update_postfix(event)
        if event['status'] in {'finished','interrupted_checkpoint'} and bar is not None:
            refresh_balance(force=True); update_postfix(event)
            bar.close(); state['bar']=None
    return callback

def provider_run_metadata(provider,historical_recovery=None):
    probe=provider.probe()
    signature={'model':probe['model'],'thinking':probe['thinking'],'response_format':probe['response_format'],'output_contract':probe['output_contract'],'context_cache':probe['context_cache'],'prompt_sha256':probe['prompt_sha256'],'operational_prompt_sha256':probe['operational_prompt_sha256'],'records_per_request':probe['records_per_request'],'label_source':probe['label_source']}
    if historical_recovery is not None: signature['historical_recovery']=historical_recovery
    return {'provider':signature,'taxonomy':'moderacion_peru_5_salidas_v2','taxonomy_version':'2.1.0'}

FLASH_RECOVERY_CHUNKS=SOURCE if PREVIOUS_PRIMARY_PATH.is_file() else HISTORICAL_CHUNKS
FLASH_RECOVERY_SOURCES=(PREVIOUS_PRIMARY_PATH,) if PREVIOUS_PRIMARY_PATH.is_file() else HISTORICAL_FLASH_SOURCES
FLASH_RECOVERY_PROMPT=PRESERVE_PROMPT_POLICY_SHA256 if PREVIOUS_PRIMARY_PATH.is_file() else HISTORICAL_PROMPT_SHA256
PRO_RECOVERY_CHUNKS=SOURCE if PREVIOUS_REVIEW_PATH.is_file() else HISTORICAL_CHUNKS
PRO_RECOVERY_SOURCES=(PREVIOUS_REVIEW_PATH,) if PREVIOUS_REVIEW_PATH.is_file() else HISTORICAL_PRO_SOURCES
PRO_RECOVERY_PROMPT=PRESERVE_PROMPT_POLICY_SHA256 if PREVIOUS_REVIEW_PATH.is_file() else HISTORICAL_PROMPT_SHA256
FLASH_HISTORY_SIGNATURE=historical_recovery_signature(FLASH_RECOVERY_CHUNKS,FLASH_RECOVERY_SOURCES,expected_model='deepseek-v4-flash',historical_prompt_sha256=FLASH_RECOVERY_PROMPT) if RECOVER_HISTORICAL else None
PRO_HISTORY_SIGNATURE=historical_recovery_signature(PRO_RECOVERY_CHUNKS,PRO_RECOVERY_SOURCES,expected_model='deepseek-v4-pro',historical_prompt_sha256=PRO_RECOVERY_PROMPT) if RECOVER_HISTORICAL else None
PRIMARY_RUN_METADATA=provider_run_metadata(primary_provider,FLASH_HISTORY_SIGNATURE)
REVIEW_RUN_METADATA=provider_run_metadata(reviewer_provider,PRO_HISTORY_SIGNATURE)
if RECOVER_HISTORICAL:
    primary_recovery=recover_historical_annotations(CHUNKS,FLASH_RECOVERY_CHUNKS,FLASH_RECOVERY_SOURCES,PRIMARY_PATH,expected_model='deepseek-v4-flash',historical_prompt_sha256=FLASH_RECOVERY_PROMPT,run_metadata=PRIMARY_RUN_METADATA)
    review_recovery=recover_historical_annotations(CHUNKS,PRO_RECOVERY_CHUNKS,PRO_RECOVERY_SOURCES,REVIEW_PATH,expected_model='deepseek-v4-pro',historical_prompt_sha256=PRO_RECOVERY_PROMPT,run_metadata=REVIEW_RUN_METADATA,label_source='llm_remote_review_historical_recovered')
    show_result('Recuperación exacta de opiniones previas',{'Flash':primary_recovery,'Pro':review_recovery,'salida_Flash_vigente':PRIMARY_PATH,'salida_Pro_vigente':REVIEW_PATH,'regla_prompt':'se conserva el prompt_sha256 original de cada opinión; solo los pendientes usan 3.2.0'},tone='success')
    if COLAB_CONTEXT is not None and AUTO_PUBLISH_CHECKPOINTS and (primary_recovery['recovered_new'] or review_recovery['recovered_new']):
        show_result('Checkpoint histórico publicado en Drive',publish_colab_outputs(COLAB_CONTEXT),tone='success')

primary_pending=primary_recovery['pending_current_after_recovery'] if RECOVER_HISTORICAL else len(CHUNKS)
# Consumo Flash medido en el histórico por cada 5 000 chunks y tasa de caché observada de 78.56%.
scale=primary_pending/5000; input_m=8.28*scale; output_m=0.724*scale; observed_cache_rate=0.7856
cost_no_cache=input_m*0.14+output_m*0.28
cost_observed_cache=input_m*((1-observed_cache_rate)*0.14+observed_cache_rate*0.0028)+output_m*0.28
show_summary('Costo Flash de lo realmente pendiente',{'total_actual':len(CHUNKS),'recuperado_o_ya_guardado':len(CHUNKS)-primary_pending,'pendiente_Flash':primary_pending,'entrada_proyectada_M':round(input_m,2),'salida_proyectada_M':round(output_m,2),'sin_caché_USD':round(cost_no_cache,2),'con_caché_histórica_78.56%_USD':round(cost_observed_cache,2),'tope_configurado_USD':MAX_PRIMARY_COST_USD},tone='success')

def checkpoint_callback_for(output):
    checkpoint_path=output.with_suffix(output.suffix+'.checkpoint.json')
    def callback(event):
        write_json_atomic(checkpoint_path,event)
        if COLAB_CONTEXT is not None and AUTO_PUBLISH_CHECKPOINTS and event['status'] in {'periodic_checkpoint','interrupted_checkpoint'}:
            publish_colab_outputs(COLAB_CONTEXT)
    return callback

def run_campaign(rows,provider,output_name,*,limit,description):
    if not provider.probe()['credential_configured']:
        raise RuntimeError('Falta DEEPSEEK_API_KEY: configúrela como variable local o secreto privado de Colab antes de etiquetar')
    output=CAMPAIGN_ROOT/output_name
    run_metadata=PRIMARY_RUN_METADATA if output_name=='primary_flash.jsonl' else REVIEW_RUN_METADATA if output_name=='review_pro.jsonl' else provider_run_metadata(provider)
    result=annotate_batched_incremental(rows,provider,output,error_path=output.with_suffix('.errors.jsonl'),limit=limit,processing_batch_size=PROCESSING_BATCH_SIZE,progress_callback=labeling_progress(description,provider),checkpoint_callback=checkpoint_callback_for(output),checkpoint_every_batches=DRIVE_CHECKPOINT_EVERY_BATCHES,run_metadata=run_metadata,quarantine_invalid_progress=True)
    try: result['account_balance']=provider.balance_summary()
    except Exception as exc: result['account_balance_error']=f'{type(exc).__name__}: {exc}'
    write_json_atomic(output.with_suffix('.result.json'),result)
    if COLAB_CONTEXT is not None and AUTO_PUBLISH_CHECKPOINTS: publish_colab_outputs(COLAB_CONTEXT)
    return output,result

## Calibración corta Flash frente a Pro

In [5]:
from moderacion_peru.labeling_calibration import select_calibration_panel,calibrate_primary_against_reviewer

PANEL_PATH=CAMPAIGN_ROOT/'calibration_panel.jsonl'
CALIBRATION_PATH=CAMPAIGN_ROOT/'calibration_flash_vs_pro.json'
if RUN_CALIBRATION:
    if PANEL_PATH.is_file():
        panel=list(tqdm(read_jsonl(PANEL_PATH),desc='Recuperando panel congelado',unit='chunk'))
        if len(panel)!=CALIBRATION_PANEL_SIZE: raise ValueError('El panel guardado no coincide con CALIBRATION_PANEL_SIZE; use otra carpeta de campaña')
    else:
        panel_progress={'bar':tqdm(total=len(CHUNKS),desc='Seleccionando panel',unit='chunk')}
        def report_panel(event):
            if event.get('advance'): panel_progress['bar'].update(event['advance'])
        panel=select_calibration_panel(CHUNKS,panel_size=CALIBRATION_PANEL_SIZE,seed=42,max_per_video=1,progress_callback=report_panel)
        panel_progress['bar'].close()
        write_jsonl_atomic(PANEL_PATH,panel)
    flash_path,flash_panel_result=run_campaign(panel,primary_provider,'calibration_flash.jsonl',limit=None,description='Calibración Flash')
    pro_path,pro_panel_result=run_campaign(panel,reviewer_provider,'calibration_pro.jsonl',limit=None,description='Calibración Pro')
    calibration=calibrate_primary_against_reviewer(read_jsonl(flash_path),read_jsonl(pro_path),minimum_auto_count=200,bootstrap_replicates=1000)
    write_json_atomic(CALIBRATION_PATH,calibration)
    show_table('Riesgo–cobertura por umbral',calibration['comparisons'],max_rows=len(calibration['comparisons']))
    show_result('Umbral operativo calibrado',calibration,tone='success' if calibration['threshold_status']=='calibrated' else 'warning')
elif CALIBRATION_PATH.is_file():
    calibration=__import__('json').loads(CALIBRATION_PATH.read_text(encoding='utf-8-sig'))
    show_table('Calibración guardada (sin repetir API)',calibration['comparisons'],max_rows=len(calibration['comparisons']))
else:
    calibration=None
    show_callout('Calibración pendiente','Active RUN_CALIBRATION=True. El panel de 1 000 es pareado por chunk y el bootstrap agrupa por video.',tone='neutral')

threshold,auto_accepted,coverage,exact_agreement,exact_lower_one_sided_95,binary_agreement,binary_lower_one_sided_95
0.7,640,0.64,0.728125,0.6982812346733331,0.9890625,0.979948410777514
0.75,640,0.64,0.728125,0.6982812346733331,0.9890625,0.979948410777514
0.8,640,0.64,0.728125,0.6982812346733331,0.9890625,0.979948410777514
0.85,638,0.638,0.7288401253918495,0.6989690070394804,0.9890282131661442,0.9798859303183013
0.9,610,0.61,0.7377049180327869,0.707405834754549,0.9901639344262295,0.9810936360034461
0.95,434,0.434,0.804147465437788,0.7709696689578416,0.9976958525345622,0.9897391109328862


## Primera pasada completa con Flash

In [6]:
if RUN_PRIMARY:
    PRIMARY_PATH,primary_result=run_campaign(CHUNKS,primary_provider,'primary_flash_v3_2.jsonl',limit=PRIMARY_LIMIT,description='Primera pasada Flash')
    show_result('Resultado Flash',primary_result,tone='success')
else:
    show_callout('Primera pasada desactivada','PRIMARY_LIMIT=None procesa todos y solo los pendientes; use 20 únicamente para un smoke mínimo. La salida reanuda por chunk_id y muestra costo, caché y saldo reales.',tone='neutral')

## Enrutamiento y revisión dirigida con Pro

In [7]:
from moderacion_peru.labeling_calibration import build_directed_review_queue
REVIEW_QUEUE_PATH=CAMPAIGN_ROOT/'directed_review_queue.jsonl'
if RUN_DIRECTED_REVIEW:
    if calibration is None or not PRIMARY_PATH.is_file():
        raise FileNotFoundError('Complete la calibración y la primera pasada antes de revisar')
    primary_rows=list(tqdm(read_jsonl(PRIMARY_PATH),desc='Cargando propuestas Flash',unit='anotación'))
    primary_ids={row['chunk_id'] for row in primary_rows}
    paired_chunks=[row for row in tqdm(CHUNKS,desc='Uniendo chunks con Flash',unit='chunk') if row['chunk_id'] in primary_ids]
    queue_progress={'bar':tqdm(total=len(paired_chunks),desc='Construyendo cola Pro',unit='chunk')}
    def report_queue(event):
        if event.get('advance'): queue_progress['bar'].update(event['advance'])
    review_queue,routing=build_directed_review_queue(paired_chunks,primary_rows,confidence_threshold=REVIEW_CONFIDENCE_THRESHOLD,safe_control_rate=SAFE_CONTROL_RATE,max_needs_review=MAX_NEEDS_REVIEW_FOR_PRO,seed=42,progress_callback=report_queue)
    queue_progress['bar'].close()
    write_jsonl_atomic(REVIEW_QUEUE_PATH,review_queue); write_json_atomic(CAMPAIGN_ROOT/'routing_summary.json',routing)
    reviewed_ids={row['chunk_id'] for row in read_jsonl(REVIEW_PATH)} if REVIEW_PATH.is_file() else set()
    pending_review_count=sum(row['chunk_id'] not in reviewed_ids for row in review_queue)
    projected_review_cost=pending_review_count*OBSERVED_REVIEW_COST_PER_1000_USD/1000
    budget_balance=reviewer_provider.balance_summary()
    show_summary('Previsión antes de enviar corpus a Pro',{'revisiones_Pro_preservadas':len(reviewed_ids),'cola_nueva_pendiente':pending_review_count,'controles_seguros_seleccionados':routing['routing_reasons'].get('safe_control',0),'costo_puntual_proyectado_USD':round(projected_review_cost,2),'tope_artificial_USD':MAX_REVIEW_COST_USD,'saldo_actual_USD':budget_balance['total_balance_usd'],'saldo_recomendado_no_bloqueante_USD':RECOMMENDED_REVIEW_START_BALANCE_USD},tone='success' if budget_balance['total_balance_usd']>=projected_review_cost else 'warning')
    if budget_balance['total_balance_usd'] < RECOMMENDED_REVIEW_START_BALANCE_USD:
        show_callout('Saldo menor que la recomendación',f"Saldo Pro US${budget_balance['total_balance_usd']:.2f}; la referencia conservadora es US${RECOMMENDED_REVIEW_START_BALANCE_USD:.2f}, pero esto no bloquea la ejecución. El proveedor se detendrá si se agota el saldo y la reanudación continuará por chunk_id.",tone='warning')
    REVIEW_PATH,review_result=run_campaign(review_queue,reviewer_provider,'review_pro_v3_2.jsonl',limit=REVIEW_LIMIT,description='Revisión dirigida Pro')
    show_summary('Enrutamiento operativo actualizado',routing,tone='success')
    show_result('Resultado Pro',review_result,tone='success')
else:
    show_callout('Revisión dirigida desactivada','Active solo después de completar Flash. La regla presupuestada revisa todo daño, las 36 000 abstenciones de menor confianza, seguros con confianza menor que 0.85 y un control seguro aleatorio reproducible del 1%.',tone='neutral')

Cargando propuestas Flash: 0anotación [00:00, ?anotación/s]

Uniendo chunks con Flash:   0%|          | 0/166940 [00:00<?, ?chunk/s]

Construyendo cola Pro:   0%|          | 0/166940 [00:00<?, ?chunk/s]

revisiones_Pro_preservadas,29270
cola_nueva_pendiente,40705
controles_seguros_seleccionados,971
costo_puntual_proyectado_USD,13.67
tope_ejecución_USD,14.5
saldo_actual_USD,15.75


Verificando progreso guardado: 0chunk [00:00, ?chunk/s]

Buscando chunks pendientes:   0%|          | 0/55424 [00:00<?, ?chunk/s]

Revisión dirigida Pro:   0%|          | 0/40705 [00:00<?, ?chunk/s]

source_chunks,166940
primary_annotations,166940
selected,55424
confidence_threshold,0.85
safe_control_rate,0.01
needs_review_candidates,52015
max_needs_review,36000
needs_review_priority,score_confianza_asc_then_seeded_sha256
seed,42
routing_reasons,"Ver detalle{ ""needs_review"": 36000, ""damage"": 18265, ""safe_control"": 971, ""low_confidence"": 188 }"


already_completed,29270
selected,40705
labeled,40704
errors,1
batches,255
request_groups,8141
elapsed_seconds,4741.879
chunks_per_minute,515.049
run_manifest,D:\trabajo_PLN\Trabajo_PLN-MIA-Grupo4\datos\etiquetado\cascada_deepseek_v4\review_pro.jsonl.run.json
quarantined_progress,—
provider_usage,"Ver detalle{ ""requests"": 8213, ""input_tokens"": 39208679, ""cache_hit_tokens"": 22078208, ""cache_miss_tokens"": 17130471, ""output_tokens"": 6808872, ""estimated_cost_usd"": 13.455507, ""cache_hit_rate"": 0.563095, ""groq_gpt_oss_20b_batch_equivalent_usd"": 2.491656, ""max_cost_usd"": 14.5, ""budget_exhausted"": false }"


## Resultados persistidos y reportables

In [8]:
import json
saved={}
for path in sorted(CAMPAIGN_ROOT.glob('*.result.json')):
    saved[path.stem.replace('.result','')]=json.loads(path.read_text(encoding='utf-8-sig'))
if CALIBRATION_PATH.is_file():
    current=json.loads(CALIBRATION_PATH.read_text(encoding='utf-8-sig'))
    show_table('Tabla reportable de calibración',current['comparisons'],max_rows=len(current['comparisons']))
    show_summary('Conclusión de calibración',{'estado':current['threshold_status'],'umbral':current['selected_threshold'],'pares':current['paired_chunks'],'referencia':current['reference_kind'],'bootstrap_agrupado_por_video':current['selected_threshold_cluster_bootstrap_95']},tone='success' if current['threshold_status']=='calibrated' else 'warning')
show_result('Resultados recuperados sin repetir cálculos',saved,tone='success' if saved else 'neutral')
show_callout('Límite inferencial','Flash–Pro es una calibración operativa y no reemplaza validación humana independiente. El score declarado no se interpreta como probabilidad estadística.',tone='warning')

threshold,auto_accepted,coverage,exact_agreement,exact_lower_one_sided_95,binary_agreement,binary_lower_one_sided_95
0.7,640,0.64,0.728125,0.6982812346733331,0.9890625,0.979948410777514
0.75,640,0.64,0.728125,0.6982812346733331,0.9890625,0.979948410777514
0.8,640,0.64,0.728125,0.6982812346733331,0.9890625,0.979948410777514
0.85,638,0.638,0.7288401253918495,0.6989690070394804,0.9890282131661442,0.9798859303183013
0.9,610,0.61,0.7377049180327869,0.707405834754549,0.9901639344262295,0.9810936360034461
0.95,434,0.434,0.804147465437788,0.7709696689578416,0.9976958525345622,0.9897391109328862


estado,inconclusive_conservative_threshold
umbral,0.95
pares,1000
referencia,stronger_llm_not_human_ground_truth
bootstrap_agrupado_por_video,"Ver detalle{ ""replicates"": 1000, ""exact_low"": 0.7649769585253456, ""exact_high"": 0.8410138248847926, ""binary_low"": 0.9930875576036866, ""binary_high"": 1.0 }"


calibration_flash,"Ver detalle{ ""already_completed"": 0, ""selected"": 1000, ""labeled"": 1000, ""errors"": 0, ""batches"": 7, ""request_groups"": 200, ""elapsed_seconds"": 97.25, ""chunks_per_minute"": 616.969, ""run_manifest"": ""D:\\trabajo_PLN\\Trabajo_PLN-MIA-Grupo4\\datos\\etiquetado\\cascada_deepseek_v4\\calibration_flash.jsonl.run.json"", ""quarantined_progress"": null, ""provider_usage"": { ""requests"": 202, ""input_tokens"": 695844, ""cache_hit_tokens"": 446848, ""cache_miss_tokens"": 248996, ""output_tokens"": 132994, ""estimated_cost_usd"": 0.073349, ""cache_hit_rate"": 0.642167, ""groq_gpt_oss_20b_batch_equivalent_usd"": 0.046043, ""max_cost_usd"": 60.0, ""budget_exhausted"": false }, ""account_balance"": { ""status"": ""balance_verified_no_corpus_sent"", ""is_available"": true, ""currency"": ""USD"", ""total_balance_usd"": 19.68, ""granted_balance_usd"": 0.0, ""topped_up_balance_usd"": 19.68 } }"
calibration_pro,"Ver detalle{ ""already_completed"": 0, ""selected"": 1000, ""labeled"": 1000, ""errors"": 0, ""batches"": 7, ""request_groups"": 200, ""elapsed_seconds"": 122.593, ""chunks_per_minute"": 489.424, ""run_manifest"": ""D:\\trabajo_PLN\\Trabajo_PLN-MIA-Grupo4\\datos\\etiquetado\\cascada_deepseek_v4\\calibration_pro.jsonl.run.json"", ""quarantined_progress"": null, ""provider_usage"": { ""requests"": 204, ""input_tokens"": 701425, ""cache_hit_tokens"": 376832, ""cache_miss_tokens"": 324593, ""output_tokens"": 145585, ""estimated_cost_usd"": 0.269223, ""cache_hit_rate"": 0.537238, ""groq_gpt_oss_20b_batch_equivalent_usd"": 0.048141, ""max_cost_usd"": 25.0, ""budget_exhausted"": false }, ""account_balance"": { ""status"": ""balance_verified_no_corpus_sent"", ""is_available"": true, ""currency"": ""USD"", ""total_balance_usd"": 19.64, ""granted_balance_usd"": 0.0, ""topped_up_balance_usd"": 19.64 } }"
primary_flash,"Ver detalle{ ""already_completed"": 166935, ""selected"": 5, ""labeled"": 5, ""errors"": 0, ""batches"": 1, ""request_groups"": 1, ""elapsed_seconds"": 4.111, ""chunks_per_minute"": 72.966, ""run_manifest"": ""D:\\trabajo_PLN\\Trabajo_PLN-MIA-Grupo4\\datos\\etiquetado\\cascada_deepseek_v4\\primary_flash.jsonl.run.json"", ""quarantined_progress"": null, ""provider_usage"": { ""requests"": 1, ""input_tokens"": 3499, ""cache_hit_tokens"": 2816, ""cache_miss_tokens"": 683, ""output_tokens"": 552, ""estimated_cost_usd"": 0.000258, ""cache_hit_rate"": 0.804801, ""groq_gpt_oss_20b_batch_equivalent_usd"": 0.000214, ""max_cost_usd"": 60.0, ""budget_exhausted"": false }, ""account_balance"": { ""status"": ""balance_verified_no_corpus_sent"", ""is_available"": true, ""currency"": ""USD"", ""total_balance_usd"": 10.48, ""granted_balance_usd"": 0.0, ""topped_up_balance_usd"": 10.48 } }"
review_pro,"Ver detalle{ ""already_completed"": 29270, ""selected"": 40705, ""labeled"": 40704, ""errors"": 1, ""batches"": 255, ""request_groups"": 8141, ""elapsed_seconds"": 4741.879, ""chunks_per_minute"": 515.049, ""run_manifest"": ""D:\\trabajo_PLN\\Trabajo_PLN-MIA-Grupo4\\datos\\etiquetado\\cascada_deepseek_v4\\review_pro.jsonl.run.json"", ""quarantined_progress"": null, ""provider_usage"": { ""requests"": 8213, ""input_tokens"": 39208679, ""cache_hit_tokens"": 22078208, ""cache_miss_tokens"": 17130471, ""output_tokens"": 6808872, ""estimated_cost_usd"": 13.455507, ""cache_hit_rate"": 0.563095, ""groq_gpt_oss_20b_batch_equivalent_usd"": 2.491656, ""max_cost_usd"": 14.5, ""budget_exhausted"": false }, ""account_balance"": { ""status"": ""balance_verified_no_corpus_sent"", ""is_available"": true, ""currency"": ""USD"", ""total_balance_usd"": 2.74, ""granted_balance_usd"": 0.0, ""topped_up_balance_usd"": 2.74 } }"


## Publicación o checkpoint en Drive

Cada época completa se guarda y verifica por separado en `trainer_checkpoints`. Al terminar una corrida 03_x se publica automáticamente el candidato final en una de dos ranuras redundantes. Esta celda permite repetir manualmente esa publicación; no vuelve a incluir los directorios transitorios de `Trainer`.

In [ ]:
PUBLISH_TO_DRIVE = False
if COLAB_CONTEXT is not None and PUBLISH_TO_DRIVE:
    from moderacion_peru.colab import publish_colab_outputs
    show_result('Publicación en Drive', publish_colab_outputs(COLAB_CONTEXT), tone='success')
elif COLAB_CONTEXT is not None and globals().get('AUTO_PUBLISH_CHECKPOINTS'):
    show_callout('Checkpoint automático activo', 'La recuperación, los checkpoints periódicos, Ctrl+C y cada cierre de campaña ya publican una copia verificable en Drive.', tone='success')
elif COLAB_CONTEXT is not None:
    show_callout('Publicación manual desactivada', 'Los entrenamientos 03_x ya publican automáticamente al completar; active esta celda solo para repetir la publicación final.', tone='neutral')
else:
    show_callout('Backend local', 'Los artefactos ya permanecen en el workspace.', tone='success')

## Referencias

[1] DeepSeek, "DeepSeek V4 Preview Release," DeepSeek API Documentation, Apr. 2026. [Online]. Available: https://api-docs.deepseek.com/news/news260424/. Accessed: Aug. 5, 2026.

[2] DeepSeek, "Models and Pricing," DeepSeek API Documentation, 2026. [Online]. Available: https://api-docs.deepseek.com/quick_start/pricing. Accessed: Aug. 7, 2026.

[3] B. Settles, "Active Learning Literature Survey," Univ. Wisconsin–Madison, Computer Sciences Tech. Rep. 1648, 2009. [Online]. Available: https://minds.wisconsin.edu/handle/1793/60660

[4] H. Schroeder, D. Roy, and J. Kabbara, "Just Put a Human in the Loop? Investigating LLM-Assisted Annotation for Subjective Tasks," in Findings ACL, 2025, pp. 25771–25795, doi: 10.18653/v1/2025.findings-acl.1323.

[5] Google Colab, "Known Issues and Workarounds," googlecolab/colab-vscode Wiki, 2026. [Online]. Available: https://github.com/googlecolab/colab-vscode/wiki/Known-Issues-and-Workarounds. Accessed: Aug. 5, 2026.

[6] National Institute of Standards and Technology, "Secure Hash Standard (SHS)," FIPS PUB 180-4, Aug. 2015, doi: 10.6028/NIST.FIPS.180-4.